In [ ]:
!pip install openai pandas numpy scikit-learn sentence-transformers

import openai
import pandas as pd
import numpy as np
import time
import re
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import pdist
from scipy.linalg import eigh

In [ ]:

# YANDEXGPT
YANDEX_CLOUD_FOLDER = "-----------"  # Замените на ваш folder_id
YANDEX_CLOUD_API_KEY = "-----------"      # Замените на ваш API ключ
YANDEX_CLOUD_MODEL = "yandexgpt-5.1/latest"

client = openai.OpenAI(
    api_key=YANDEX_CLOUD_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
    project=YANDEX_CLOUD_FOLDER
)

# Эмбеддинги
embedder = SentenceTransformer('intfloat/multilingual-e5-small') # slava
#embedder = SentenceTransformer('all-MiniLM-L6-v2') #tqa

#  ЗАПРОС К YANDEXGPT
def ask_yandex_gpt(prompt, temperature=0.7, max_tokens=256):
    try:
        response = client.responses.create(
            model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
            temperature=temperature,
            instructions="You are a helpful assistant. Answer questions accurately and concisely. Always answer in the same language as the question.",
            input=prompt,
            max_output_tokens=max_tokens
        )
        return response.output_text.strip()
    except Exception as e:
        print(f"Ошибка API: {e}")
        return ""



In [ ]:
def generate_responses_yandex(question, context=None, num_responses=5, temperature=0.7, max_tokens=256):

    if context:
        prompt = f"""Context: {context}

Question: {question}

Answer in the same language as the question. Answer:"""
    else:
        prompt = f"""Question: {question}

Answer in the same language as the question. Answer:"""

    responses = []
    for i in range(num_responses):
        if i > 0:
            time.sleep(0.5)

        answer = ask_yandex_gpt(prompt, temperature=temperature, max_tokens=max_tokens)
        if answer:
            responses.append(answer.strip())
        else:
            responses.append("[Error generating response]")

        print(f"  Generation {i+1}/{num_responses} completed")

    return responses



In [ ]:

# КЛАСС ДЕТЕКТОРА ДЛЯ YANDEXGPT

class HallucinationDetectorYandex:
    def __init__(self, embedder):
        self.embedder = embedder

        # Пороги для метрик(По умолчанию)
        self.thresholds = {
            'semantic_entropy': 0.4,
            'eigen_score': 0.3,
            'sar_score': 0.6,
            'num_sets': 2,
            'self_confidence': 0.5,
            'verbalized_uncertainty': 0.5,
            'embedding_similarity': 0.7
        }

    def generate_responses(self, question, context=None, num_responses=5, temperature=0.7):
        return generate_responses_yandex(question, context, num_responses, temperature)

    def semantic_entropy(self, responses):
        if len(responses) < 2:
            return 0.0
        emb = self.embedder.encode(responses)
        sim_matrix = cosine_similarity(emb)
        np.fill_diagonal(sim_matrix, 0)
        mean_sim = sim_matrix.sum() / (len(responses) * (len(responses) - 1))
        return 1.0 - mean_sim

    def eigen_score(self, responses):
        if len(responses) < 3:
            return 0.0

        emb = self.embedder.encode(responses)
        emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
        gram_matrix = np.dot(emb, emb.T)
        eigenvalues = eigh(gram_matrix, eigvals_only=True)
        eigenvalues = eigenvalues[eigenvalues > 1e-10]

        if len(eigenvalues) == 0:
            return 0.0

        eigenvalues = eigenvalues / np.sum(eigenvalues)
        eigen_entropy = -np.sum(eigenvalues * np.log(eigenvalues + 1e-10))
        max_entropy = np.log(min(len(responses), len(eigenvalues)))
        if max_entropy > 0:
            eigen_entropy = eigen_entropy / max_entropy

        return eigen_entropy

    def sar_score(self, responses):
        if len(responses) < 2:
            return 1.0

        emb = self.embedder.encode(responses)
        distances = pdist(emb, metric='cosine')
        if len(distances) == 0:
            return 1.0

        similarities = 1 - distances
        return np.median(similarities)

    def num_sets(self, responses, eps=0.3, min_samples=1):
        if len(responses) < 2:
            return 1

        emb = self.embedder.encode(responses)
        clustering = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine').fit(emb)
        labels = clustering.labels_
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        return max(1, n_clusters)

    def self_check(self, question, answer):
        prompt = f"""Question: {question}
Proposed answer: {answer}

On a scale from 0 to 1, how confident are you that this answer is correct?
0 = not confident at all, 1 = completely confident.
Answer with only a number (e.g., 0.85)."""

        response = ask_yandex_gpt(prompt, temperature=0.0)

        match = re.search(r'0\.\d+|1\.0|0|1', response)
        if match:
            confidence = float(match.group())
        else:
            confidence = 0.5
        return confidence

    def verbalized_uncertainty(self, question, answer):
        prompt = f"""Question: {question}
Answer: {answer}

How confident are you in this answer? Choose one:
A) Very confident
B) Somewhat confident
C) Not very confident
D) Not confident at all

Respond with only the letter A, B, C, or D."""

        response = ask_yandex_gpt(prompt, temperature=0.0)

        mapping = {'A': 1.0, 'B': 0.7, 'C': 0.3, 'D': 0.0}
        for letter in mapping:
            if letter in response.upper():
                return mapping[letter]
        return 0.5

    def compare_with_embedding(self, model_answer, correct_answer):
        emb1 = self.embedder.encode([model_answer])[0]
        emb2 = self.embedder.encode([correct_answer])[0]
        similarity = cosine_similarity([emb1], [emb2])[0][0]
        is_match = similarity >= self.thresholds['embedding_similarity']
        return similarity, is_match

    def interpret_metric(self, metric_name, value):
        threshold = self.thresholds.get(metric_name, 0.5)

        if metric_name in ['self_confidence', 'sar_score', 'verbalized_uncertainty']:
            is_hallucination = value < threshold
        elif metric_name == 'num_sets':
            is_hallucination = value >= threshold
        else:
            is_hallucination = value > threshold

        return is_hallucination

    def evaluate(self, question, correct_answer=None, context=None, num_responses=5):
        print(f"\n🔍 QUESTION: {question[:80]}...")

        # Generate responses
        responses = self.generate_responses(question, context, num_responses=num_responses)
        main_answer = responses[0] if responses else ""

        print(f"💬 ANSWER: {main_answer[:150]}...")

        result = {
            'question': question,
            'correct_answer': correct_answer,
            'model_answer': main_answer,
            'metrics': {},
            'hallucination_flags': {}
        }

        # Calculate metrics
        if len(responses) >= 2:
            result['metrics']['semantic_entropy'] = self.semantic_entropy(responses)
            result['hallucination_flags']['semantic_entropy'] = self.interpret_metric('semantic_entropy', result['metrics']['semantic_entropy'])

            result['metrics']['eigen_score'] = self.eigen_score(responses)
            result['hallucination_flags']['eigen_score'] = self.interpret_metric('eigen_score', result['metrics']['eigen_score'])

            result['metrics']['sar_score'] = self.sar_score(responses)
            result['hallucination_flags']['sar_score'] = self.interpret_metric('sar_score', result['metrics']['sar_score'])

            result['metrics']['num_sets'] = self.num_sets(responses)
            result['hallucination_flags']['num_sets'] = self.interpret_metric('num_sets', result['metrics']['num_sets'])

        result['metrics']['self_confidence'] = self.self_check(question, main_answer)
        result['hallucination_flags']['self_confidence'] = self.interpret_metric('self_confidence', result['metrics']['self_confidence'])

        result['metrics']['verbalized_uncertainty'] = self.verbalized_uncertainty(question, main_answer)
        result['hallucination_flags']['verbalized_uncertainty'] = self.interpret_metric('verbalized_uncertainty', result['metrics']['verbalized_uncertainty'])

        # Voting
        if result['hallucination_flags']:
            votes = sum(result['hallucination_flags'].values())
            total = len(result['hallucination_flags'])
            ratio = votes / total

            if ratio > 0.6:
                result['voting_result'] = "HALLUCINATION"
            elif ratio > 0.4:
                result['voting_result'] = "UNCERTAIN"
            else:
                result['voting_result'] = "NORMAL"

            result['vote_ratio'] = ratio
            result['hallucination_votes'] = votes
            result['total_votes'] = total

        # Verification
        if correct_answer:
            emb_sim, emb_match = self.compare_with_embedding(main_answer, correct_answer)
            result['embedding_similarity'] = emb_sim
            result['embedding_match'] = emb_match

        return result

# Инициализация
detector = HallucinationDetectorYandex(embedder)


In [ ]:
"""
#TruthfulQA dataset
url = "https://raw.githubusercontent.com/sylinrl/TruthfulQA/main/TruthfulQA.csv"
response = requests.get(url)

if response.status_code == 200:
    df_truthfulqa = pd.read_csv(StringIO(response.text))
    print(f"📊 Размер: {df_truthfulqa.shape[0]} строк, {df_truthfulqa.shape[1]} колонок")
else:
    print(f"❌ Ошибка загрузки: {response.status_code}")

def create_test_cases_from_truthfulqa(df, max_samples=None):

    test_cases = []

    if max_samples:
        df_filtered = df.head(max_samples)
    else:
        df_filtered = df

    for idx, row in df_filtered.iterrows():
        question = row['Question']
        if pd.notna(row['Best Answer']):
            correct = row['Best Answer']
        elif pd.notna(row['Correct Answers']):
            correct = str(row['Correct Answers']).split(';')[0].strip()
        else:
            correct = ""
        test_case = {
            'question': question,
            'correct': correct
        }
        test_cases.append(test_case)

    return test_cases



test_cases_truthfulqa = create_test_cases_from_truthfulqa(
    df_truthfulqa,
    max_samples=None
)

print(f"✅ Создано {len(test_cases_truthfulqa)} тестовых кейсов")


print("\n" + "="*60)
print("ПРИМЕРЫ ТЕСТОВЫХ КЕЙСОВ :")
print("="*60)

for i, case in enumerate(test_cases_truthfulqa):
    print(f"\n--- Кейс {i+1} ---")
    print(f"Вопрос: {case['question']}")
    print(f"Правильный ответ: {case['correct']}")



import json
with open('truthfulqa_test_cases.json', 'w', encoding='utf-8') as f:
    json.dump(test_cases_truthfulqa, f, ensure_ascii=False, indent=2)


print(f"📁 Тестовые кейсы сохранены в 'truthfulqa_test_cases.json'")
print(f"📊 Всего кейсов: {len(test_cases_truthfulqa)}")
"""

In [ ]:
import json
#slava dataset
# Загружаем тест-кейсы из JSON файла
with open('test_cases_slava.json', 'r', encoding='utf-8') as f:
    test_cases_slava = json.load(f)

print(f"Загружено {len(test_cases_slava)} тест-кейсов")

# Проверяем первый кейс
print("\nПервый загруженный кейс:")
print(f"Вопрос: {test_cases_slava[0]['question'][:100]}...")
print(f"Ответ: {test_cases_slava[0]['correct']}")

Загружено 500 тест-кейсов

Первый загруженный кейс:
Вопрос: Прочитайте приведённую далее задачу и выполните по ней задание. Задача: В каком году произошла битва...
Ответ: 3


In [ ]:
def run_batch_evaluation(test_cases, detector, num_responses=3):
    results = []

    for i, test in enumerate(test_cases):
        print(f"\n[{i+1}/{len(test_cases)}] Processing: {test['question'][:60]}...")

        result = detector.evaluate(
            question=test["question"],
            correct_answer=test.get("correct"),
            num_responses=num_responses
        )

        row = {
            'Question': result['question'],
            'Correct Answer': result['correct_answer'],
            'Model Answer': result['model_answer'],
            'Semantic Entropy': result['metrics'].get('semantic_entropy', None),
            'EigenScore': result['metrics'].get('eigen_score', None),
            'SAR Score': result['metrics'].get('sar_score', None),
            'NumSets': result['metrics'].get('num_sets', None),
            'Self Confidence': result['metrics'].get('self_confidence', None),
            'Verbalized Uncertainty': result['metrics'].get('verbalized_uncertainty', None),
            'Voting Result': result.get('voting_result', ''),
            'Hallucination Votes': f"{result.get('hallucination_votes', 0)}/{result.get('total_votes', 0)}",
        }

        if 'embedding_similarity' in result:
            row['Similarity with Correct'] = result['embedding_similarity']

        results.append(row)

        # Save intermediate results
        pd.DataFrame(results).to_csv('yandex_gpt_truthfulqa_results.csv', index=False, encoding='utf-8')
        print(f"  → Intermediate saved ({len(results)} examples)")

    return pd.DataFrame(results)




In [ ]:
df_results = run_batch_evaluation(test_cases_slava[300:], detector, num_responses=5)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
display(df_results)

# Save final results
df_results.to_csv('yandex_gpt_slava_300.csv', index=False, encoding='utf-8')





[1/200] Processing: Прочитайте приведённую далее задачу и выполните по ней задан...

🔍 QUESTION: Прочитайте приведённую далее задачу и выполните по ней задание. Задача: С какой ...
  Generation 1/5 completed
  Generation 2/5 completed
  Generation 3/5 completed
  Generation 4/5 completed
  Generation 5/5 completed
💬 ANSWER: 1...
  → Intermediate saved (1 examples)

[2/200] Processing: Прочитайте приведённую далее задачу и выполните по ней задан...

🔍 QUESTION: Прочитайте приведённую далее задачу и выполните по ней задание. Задача: В какой ...
  Generation 1/5 completed
  Generation 2/5 completed
  Generation 3/5 completed
  Generation 4/5 completed
  Generation 5/5 completed
💬 ANSWER: 3...
  → Intermediate saved (2 examples)

[3/200] Processing: Прочитайте приведённую далее задачу и выполните по ней задан...

🔍 QUESTION: Прочитайте приведённую далее задачу и выполните по ней задание. Задача: Располож...
  Generation 1/5 completed
  Generation 2/5 completed
  Generation 3/5 completed
 

,Question,Correct Answer,Model Answer,Semantic Entropy,EigenScore,SAR Score,NumSets,Self Confidence,Verbalized Uncertainty,Voting Result,Hallucination Votes,Similarity with Correct
0,Прочитайте приведённую далее задачу и выполнит...,1,1,-2.384186e-07,1.253029e-06,1.0,1,1.00,1.0,NORMAL,0/6,1.000000
1,Прочитайте приведённую далее задачу и выполнит...,3,3,-2.384186e-07,9.906479e-07,1.0,1,0.80,1.0,NORMAL,0/6,1.000000
2,Прочитайте приведённую далее задачу и выполнит...,312,312,0.000000e+00,1.947218e-06,1.0,1,1.00,1.0,NORMAL,0/6,1.000000
3,"Прочитайте приведённые далее задачу, текст и в...",324,324,0.000000e+00,1.721040e-06,1.0,1,1.00,1.0,NORMAL,0/6,1.000000
4,"Прочитайте приведённые далее задачу, текст и в...",4,4,0.000000e+00,1.404331e-06,1.0,1,0.75,0.3,NORMAL,1/6,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
195,Прочитайте приведённую далее задачу и выполнит...,41325,43125,-2.384186e-07,1.013993e-06,1.0,1,1.00,1.0,NORMAL,0/6,0.930364
196,Прочитайте приведённую далее задачу и выполнит...,4,4,0.000000e+00,1.404331e-06,1.0,1,1.00,1.0,NORMAL,0/6,1.000000
197,Прочитайте приведённую далее задачу и выполнит...,245,245,-2.384186e-07,1.227116e-06,1.0,1,1.00,1.0,NORMAL,0/6,1.000000
198,Прочитайте приведённую далее задачу и выполнит...,3,3,-2.384186e-07,9.906479e-07,1.0,1,1.00,1.0,NORMAL,0/6,1.000000
